In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: GPU is not available")

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [ ]:
!pip install -q transformers datasets sentencepiece sacrebleu tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.0/129.0 kB 6.7 MB/s eta 0:00:00


In [ ]:
!pip install -q pandas==2.2.3

In [ ]:
import torch
import pandas as pd
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sacrebleu import sentence_bleu
from tqdm.auto import tqdm

In [ ]:
dataset = load_dataset("sanganaka/ramayana-anvaya")
print(dataset)

README.md:   0%|          | 0.00/408 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.62MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  401kB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/16447 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1829 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sloka', 'prose'],
        num_rows: 16447
    })
    test: Dataset({
        features: ['sloka', 'prose'],
        num_rows: 1829
    })
})


In [ ]:
test_dataset = dataset["test"]
print("Number of test samples:", len(test_dataset))

Number of test samples: 1829


In [ ]:
print(test_dataset.column_names)

['sloka', 'prose']


In [ ]:
print(test_dataset[0])

{'sloka': 'धर्मस्यपुत्रोबलवान्सुषेणइतिविश्रुतः सविद्युन्मालिनासार्धमयुध्यतमहाकपिः वानराश्चापरेभीमाराक्षसैरपरैस्सह द्वन्द्वंसमीयुस्सहसायुद्धायबहुभिस्सह', 'prose': 'धर्मस्यपुत्रो बलवान्सुषेण इति विश्रुतः सविद्युन्मालिना सार्धमयुध्यत महाकपिः घोराः अपरे वानराश्च बहुभिःसह युद्धायच अपरैः राक्षसैःसह द्वन्द्वंसमीयुः'}


In [ ]:
columns = test_dataset.column_names
print("Available columns:")
for i, col in enumerate(columns):
    print(i, ":", col)

Available columns:
0 : sloka
1 : prose


In [ ]:
sample = test_dataset[0]
for key, value in sample.items():
    print("\nCOLUMN:", key)
    print("VALUE:", value)


COLUMN: sloka
VALUE: धर्मस्यपुत्रोबलवान्सुषेणइतिविश्रुतः सविद्युन्मालिनासार्धमयुध्यतमहाकपिः वानराश्चापरेभीमाराक्षसैरपरैस्सह द्वन्द्वंसमीयुस्सहसायुद्धायबहुभिस्सह

COLUMN: prose
VALUE: धर्मस्यपुत्रो बलवान्सुषेण इति विश्रुतः सविद्युन्मालिना सार्धमयुध्यत महाकपिः घोराः अपरे वानराश्च बहुभिःसह युद्धायच अपरैः राक्षसैःसह द्वन्द्वंसमीयुः


In [ ]:
SLOKA_COLUMN = "sloka"
PROSE_COLUMN = "prose"

In [ ]:
SLOKA_COLUMN = "YOUR_SLOKA_COLUMN"
PROSE_COLUMN = "YOUR_PROSE_COLUMN"

In [ ]:
MODEL_NAME = "facebook/nllb-200-distilled-600M"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    src_lang="san_Deva"
)

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 4.85MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.3MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/3.55k [00:00<?, ?B/s]

In [ ]:
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME
)
model = model.to(device)
model.eval()
print("NLLB model loaded successfully.")

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 2.46GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.46GB            

model.safetensors: downloading bytes:           |  0.00B            

NLLB model loaded successfully.


In [ ]:
SOURCE_LANG = "san_Deva"
TARGET_LANG = "eng_Latn"
target_token_id = tokenizer.convert_tokens_to_ids(TARGET_LANG)
print("Source language:", SOURCE_LANG)
print("Target language:", TARGET_LANG)
print("Target token ID:", target_token_id)

Source language: san_Deva
Target language: eng_Latn
Target token ID: 256047


In [ ]:
def translate_sanskrit(text, max_length=256):
    """Translate Sanskrit text into English using NLLB."""
    if not text:
        return ""
    text = str(text).strip()
    if not text:
        return ""
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=max_length
    )
    inputs = {key: value.to(device) for key, value in inputs.items()}

    with torch.no_grad():
        output = model.generate(
            **inputs,
            forced_bos_token_id=target_token_id,
            max_length=max_length,
            num_beams=4
        )
    translation = tokenizer.batch_decode(
        output,
        skip_special_tokens=True
    )[0]
    return translation.strip()

In [ ]:
print(test_dataset.column_names)

['sloka', 'prose']


In [ ]:
print(test_dataset[0])

{'sloka': 'धर्मस्यपुत्रोबलवान्सुषेणइतिविश्रुतः सविद्युन्मालिनासार्धमयुध्यतमहाकपिः वानराश्चापरेभीमाराक्षसैरपरैस्सह द्वन्द्वंसमीयुस्सहसायुद्धायबहुभिस्सह', 'prose': 'धर्मस्यपुत्रो बलवान्सुषेण इति विश्रुतः सविद्युन्मालिना सार्धमयुध्यत महाकपिः घोराः अपरे वानराश्च बहुभिःसह युद्धायच अपरैः राक्षसैःसह द्वन्द्वंसमीयुः'}


In [ ]:
for column in test_dataset.column_names:
    print("=" * 80)
    print("COLUMN:", column)
    print("VALUE:")
    print(test_dataset[0][column])

COLUMN: sloka
VALUE:
धर्मस्यपुत्रोबलवान्सुषेणइतिविश्रुतः सविद्युन्मालिनासार्धमयुध्यतमहाकपिः वानराश्चापरेभीमाराक्षसैरपरैस्सह द्वन्द्वंसमीयुस्सहसायुद्धायबहुभिस्सह
COLUMN: prose
VALUE:
धर्मस्यपुत्रो बलवान्सुषेण इति विश्रुतः सविद्युन्मालिना सार्धमयुध्यत महाकपिः घोराः अपरे वानराश्च बहुभिःसह युद्धायच अपरैः राक्षसैःसह द्वन्द्वंसमीयुः


In [ ]:
print(test_dataset.column_names)
print(test_dataset[0])

['sloka', 'prose']
{'sloka': 'धर्मस्यपुत्रोबलवान्सुषेणइतिविश्रुतः सविद्युन्मालिनासार्धमयुध्यतमहाकपिः वानराश्चापरेभीमाराक्षसैरपरैस्सह द्वन्द्वंसमीयुस्सहसायुद्धायबहुभिस्सह', 'prose': 'धर्मस्यपुत्रो बलवान्सुषेण इति विश्रुतः सविद्युन्मालिना सार्धमयुध्यत महाकपिः घोराः अपरे वानराश्च बहुभिःसह युद्धायच अपरैः राक्षसैःसह द्वन्द्वंसमीयुः'}


In [ ]:
SLOKA_COLUMN = "actual_sloka_column"
PROSE_COLUMN = "actual_prose_column"

In [ ]:
def translate_batch(texts, batch_size=8, max_length=256):
    translations = []


    for start in tqdm(
        range(0, len(texts), batch_size),
        desc="Translating"
    ):
        batch = texts[start:start + batch_size]

        clean_batch = [
            "" if x is None else str(x)
            for x in batch
        ]

        inputs = tokenizer(
            clean_batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_length
        )

        inputs = {
            key: value.to(device)
            for key, value in inputs.items()
        }

        with torch.no_grad():
            translated_tokens = model.generate(
                **inputs,
                forced_bos_token_id=target_token_id,
                max_length=max_length,
                num_beams=4
            )

        batch_translations = tokenizer.batch_decode(
            translated_tokens,
            skip_special_tokens=True
        )

        translations.extend(
            [text.strip() for text in batch_translations]
        )

    return translations

In [ ]:
for column in test_dataset.column_names:
    print("=" * 80)
    print("COLUMN:", column)
    print("FIRST VALUE:")
    print(test_dataset[0][column])

COLUMN: sloka
FIRST VALUE:
धर्मस्यपुत्रोबलवान्सुषेणइतिविश्रुतः सविद्युन्मालिनासार्धमयुध्यतमहाकपिः वानराश्चापरेभीमाराक्षसैरपरैस्सह द्वन्द्वंसमीयुस्सहसायुद्धायबहुभिस्सह
COLUMN: prose
FIRST VALUE:
धर्मस्यपुत्रो बलवान्सुषेण इति विश्रुतः सविद्युन्मालिना सार्धमयुध्यत महाकपिः घोराः अपरे वानराश्च बहुभिःसह युद्धायच अपरैः राक्षसैःसह द्वन्द्वंसमीयुः


In [ ]:
SLOKA_COLUMN = "sloka"
PROSE_COLUMN = "prose"

In [ ]:
slokas = test_dataset[SLOKA_COLUMN]
proses = test_dataset[PROSE_COLUMN]

print("Number of test samples:", len(test_dataset))
print("Number of slokas:", len(slokas))
print("Number of prose:", len(proses))

Number of test samples: 1829
Number of slokas: 1829
Number of prose: 1829


In [ ]:
for i in range(3):
    print("=" * 80)
    print("SAMPLE", i + 1)

    print("\nSLOKA:")
    print(slokas[i])

    print("\nPROSE:")
    print(proses[i])

SAMPLE 1

SLOKA:
धर्मस्यपुत्रोबलवान्सुषेणइतिविश्रुतः सविद्युन्मालिनासार्धमयुध्यतमहाकपिः वानराश्चापरेभीमाराक्षसैरपरैस्सह द्वन्द्वंसमीयुस्सहसायुद्धायबहुभिस्सह

PROSE:
धर्मस्यपुत्रो बलवान्सुषेण इति विश्रुतः सविद्युन्मालिना सार्धमयुध्यत महाकपिः घोराः अपरे वानराश्च बहुभिःसह युद्धायच अपरैः राक्षसैःसह द्वन्द्वंसमीयुः
SAMPLE 2

SLOKA:
शबला सा रुदन्ती च क्रोशन्ती चेदमब्रवीत्वसिष्ठस्याग्रतस्स्थित्वा मेघदुन्दुभिराविणी

PROSE:
सा शबला रुदन्ती च क्रोशन्ती च वसिष्ठस्य अग्रत: स्थित्वा मेघदुन्दुभिराविणी इदम् अब्रवीत्
SAMPLE 3

SLOKA:
तप्तकाञ्चनवर्णाभा रक्ततुङ्गनखी शुभासीता नाम वरारोहा वैदेही तनुमध्यमा

PROSE:
तप्तकाञ्चनवर्णाभा रक्ततुङ्गनखी शुभा वैदेही तनुमध्यमा सीता नाम वरारोहा


In [ ]:
print("Starting Sloka translation...")

english_slokas = translate_batch(
    slokas,
    batch_size=8,
    max_length=256
)

print("\nSloka translation completed!")
print("Number of translations:", len(english_slokas))

Starting Sloka translation...


Translating:   0%|          | 0/229 [00:00<?, ?it/s]


Sloka translation completed!
Number of translations: 1829


In [ ]:
for i in range(3):
    print("=" * 80)
    print("SAMPLE", i + 1)
    print("\nSanskrit Sloka:")
    print(slokas[i])
    print("\nEnglish Sloka:")
    print(english_slokas[i])

SAMPLE 1

Sanskrit Sloka:
धर्मस्यपुत्रोबलवान्सुषेणइतिविश्रुतः सविद्युन्मालिनासार्धमयुध्यतमहाकपिः वानराश्चापरेभीमाराक्षसैरपरैस्सह द्वन्द्वंसमीयुस्सहसायुद्धायबहुभिस्सह

English Sloka:
When the Son of God heard the sound of God's trumpet, they were afraid to go to war with the devil, the devil, the serpent, the beast, the dragon, and the dragon. They were afraid to go to war with the world and with the devil.
SAMPLE 2

Sanskrit Sloka:
शबला सा रुदन्ती च क्रोशन्ती चेदमब्रवीत्वसिष्ठस्याग्रतस्स्थित्वा मेघदुन्दुभिराविणी

English Sloka:
She wept and wept, and she was filled with fury.
SAMPLE 3

Sanskrit Sloka:
तप्तकाञ्चनवर्णाभा रक्ततुङ्गनखी शुभासीता नाम वरारोहा वैदेही तनुमध्यमा

English Sloka:
and seven cups of pure blood. The cup is called the "Wrath of God " and is set in the midst of them.


In [ ]:
print("Starting Prose translation...")
english_proses = translate_batch(
    proses,
    batch_size=8,
    max_length=256
)
print("\nProse translation completed!")
print("Number of translations:", len(english_proses))

Starting Prose translation...


Translating:   0%|          | 0/229 [00:00<?, ?it/s]


Prose translation completed!
Number of translations: 1829


In [ ]:
for i in range(3):
    print("=" * 80)
    print("SAMPLE", i + 1)

    print("\nSanskrit Prose:")
    print(proses[i])

    print("\nEnglish Prose:")
    print(english_proses[i])

SAMPLE 1

Sanskrit Prose:
धर्मस्यपुत्रो बलवान्सुषेण इति विश्रुतः सविद्युन्मालिना सार्धमयुध्यत महाकपिः घोराः अपरे वानराश्च बहुभिःसह युद्धायच अपरैः राक्षसैःसह द्वन्द्वंसमीयुः

English Prose:
I have heard that the Son of God is coming with great power. He is at war with great horsemen and devils, with horses and wild beasts, and with many other demons.
SAMPLE 2

Sanskrit Prose:
सा शबला रुदन्ती च क्रोशन्ती च वसिष्ठस्य अग्रत: स्थित्वा मेघदुन्दुभिराविणी इदम् अब्रवीत्

English Prose:
And she stood before him, weeping and gnashing of teeth.
SAMPLE 3

Sanskrit Prose:
तप्तकाञ्चनवर्णाभा रक्ततुङ्गनखी शुभा वैदेही तनुमध्यमा सीता नाम वरारोहा

English Prose:
There were seven bowls full of blood, and in the midst of them there stood a man named Cethas.


In [ ]:
from sacrebleu import sentence_bleu

def calculate_bleu(candidate, reference):

    if not candidate or not reference:
        return 0.0

    score = sentence_bleu(
        candidate,
        [reference]
    )

    return score.score

In [ ]:
bleu_scores = []
for sloka_en, prose_en in tqdm(
    zip(english_slokas, english_proses),
    total=len(english_slokas),
    desc="Calculating BLEU"
):
    score = calculate_bleu(
        sloka_en,
        prose_en
    )

    bleu_scores.append(score)
print("BLEU calculation completed!")
print("Number of BLEU scores:", len(bleu_scores))

Calculating BLEU:   0%|          | 0/1829 [00:00<?, ?it/s]

BLEU calculation completed!
Number of BLEU scores: 1829


In [ ]:
for i in range(10):
    print(
        f"Sample {i+1}: BLEU = {bleu_scores[i]:.4f}"
    )

Sample 1: BLEU = 6.9784
Sample 2: BLEU = 4.7892
Sample 3: BLEU = 16.0820
Sample 4: BLEU = 3.9614
Sample 5: BLEU = 8.0612
Sample 6: BLEU = 1.3867
Sample 7: BLEU = 19.4221
Sample 8: BLEU = 4.9361
Sample 9: BLEU = 2.7157
Sample 10: BLEU = 0.0772


In [ ]:
results_df = pd.DataFrame({
    "sample_id": range(1, len(test_dataset) + 1),

    "sanskrit_sloka": slokas,

    "sanskrit_prose": proses,

    "english_sloka": english_slokas,

    "english_prose": english_proses,

    "bleu_score": bleu_scores
})

print("Rows:", len(results_df))
print("Columns:", results_df.columns.tolist())

Rows: 1829
Columns: ['sample_id', 'sanskrit_sloka', 'sanskrit_prose', 'english_sloka', 'english_prose', 'bleu_score']


In [ ]:
results_df.head()

,sample_id,sanskrit_sloka,sanskrit_prose,english_sloka,english_prose,bleu_score
0,1,धर्मस्यपुत्रोबलवान्सुषेणइतिविश्रुतः सविद्युन्म...,धर्मस्यपुत्रो बलवान्सुषेण इति विश्रुतः सविद्यु...,When the Son of God heard the sound of God's t...,I have heard that the Son of God is coming wit...,6.978424
1,2,शबला सा रुदन्ती च क्रोशन्ती चेदमब्रवीत्वसिष्ठस...,सा शबला रुदन्ती च क्रोशन्ती च वसिष्ठस्य अग्रत:...,"She wept and wept, and she was filled with fury.","And she stood before him, weeping and gnashing...",4.789232
2,3,तप्तकाञ्चनवर्णाभा रक्ततुङ्गनखी शुभासीता नाम वर...,तप्तकाञ्चनवर्णाभा रक्ततुङ्गनखी शुभा वैदेही तनु...,and seven cups of pure blood. The cup is calle...,"There were seven bowls full of blood, and in t...",16.081987
3,4,स कृत्वा भैरवं नादं चालयन्निव मेदिनीम्अङ्केनाद...,ततः सः मेदिनीम् चालयन्निव भैरवम् नादम् कृत्वा ...,"So he went to Bethlehem, in the valley of the ...","Then he called out in a loud voice: ""Go to the...",3.961394
4,5,तस्य पाण्डुरमाजह्रुश्छत्रं हेमपरिष्कृतम् शुक्ल...,पाण्डुरम् हेमपरिष्कृतम् छत्रम् हेमदण्डे यशस्कर...,"She was dressed in fine linen, purple and sapp...","These were gold, gold, copper, bronze, bronze,...",8.061204


In [ ]:
print("Missing values:")
print(results_df.isnull().sum())

Missing values:
sample_id         0
sanskrit_sloka    0
sanskrit_prose    0
english_sloka     0
english_prose     0
bleu_score        0
dtype: int64


In [ ]:
print(
    "Empty English Sloka translations:",
    (results_df["english_sloka"].str.strip() == "").sum()
)

print(
    "Empty English Prose translations:",
    (results_df["english_prose"].str.strip() == "").sum()
)

Empty English Sloka translations: 0
Empty English Prose translations: 0


In [ ]:
print("BLEU Statistics")
print("=" * 50)

print("Number of samples:", len(results_df))
print("Average BLEU:", results_df["bleu_score"].mean())
print("Minimum BLEU:", results_df["bleu_score"].min())
print("Maximum BLEU:", results_df["bleu_score"].max())
print("Median BLEU:", results_df["bleu_score"].median())

BLEU Statistics
Number of samples: 1829
Average BLEU: 6.437210279237078
Minimum BLEU: 0.0
Maximum BLEU: 63.7346589279761
Median BLEU: 3.657015913414383


In [ ]:
results_df.sort_values(
    by="bleu_score",
    ascending=False
)[
    [
        "sample_id",
        "english_sloka",
        "english_prose",
        "bleu_score"
    ]
].head(10)

,sample_id,english_sloka,english_prose,bleu_score
693,694,When the Holy Spirit had eaten the fruit of th...,"When the Spirit of righteousness came, he ate ...",63.734659
699,700,"He was a man of great power, a man of great po...","He was a man of great power, a man of great po...",62.894322
1468,1469,And it shall come to pass at the end of the th...,And it shall come to pass at the end of the th...,60.630005
157,158,"When he saw them, he was amazed and amazed.","When he saw them, he was terrified.",59.004687
1228,1229,"When I heard this, I quickly left the palace a...","When I heard about this, I left the palace and...",58.569555
788,789,"When the high priest, who belonged to the patr...","When the high priest, who belonged to the patr...",57.383289
1308,1309,He had a crown of gold and a crown of purple a...,He had a crown of gold and a shadow like a kin...,54.374277
1421,1422,He took the money and bowed his head to the go...,Then he took a piece of plum money and bowed h...,52.253560
851,852,"When he heard this, he was filled with grief a...","When he had said this, he was filled with grie...",50.820276
188,189,"When the king heard this, he was troubled and ...","When the king heard this, he was troubled, won...",50.435455


In [ ]:
results_df.sort_values(
    by="bleu_score",
    ascending=True
)[
    [
        "sample_id",
        "english_sloka",
        "english_prose",
        "bleu_score"
    ]
].head(10)

,sample_id,english_sloka,english_prose,bleu_score
347,348,The three - - - - - - - - - - - - - - - - - - ...,Ra'ah: Here: the bridge is as: Bhadith: his: R...,0.0
630,631,"For thou hast spoken the truth, and hast led t...",Location: Location: Location: Location: Locati...,0.0
1714,1715,and his words are sweet and gracious to those ...,"The serpent, the great serpent, the righteous ...",0.0
1746,1747,"""A good king is honored with good deeds; a str...","The kings, the mighty, the righteous, the migh...",0.0
1687,1688,"Do not be silent in your congregation, but spe...",Don't worry about what we're going to say. The...,0.0
881,882,"The Lord is faithful, and the Lord is faithful...",Mother: Friend: Faithful: Faithful: Faithful: ...,0.0
686,687,"""Mother, honey, honey, honey, honey, honey, ho...",Madara Madara Mara Mara Mara Mara Mara Mara Ma...,0.0
1350,1351,"""This is the voice of the lambs, the voice of ...",Sheep: Sheep: Sheep: Sheep: Sheep: Sheep: Shee...,0.0
1607,1608,But you will be like a wolf in sheep's clothin...,"Depraved men, wild beasts, wise men, deceived ...",0.0
795,796,"They must be patient, patient, patient, patien...",The crude winds are united: united: united: un...,0.0


In [ ]:
OUTPUT_FILE = "ramayana_anvaya_nllb_bleu_results.csv"

results_df.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)

print("CSV created successfully!")
print(OUTPUT_FILE)

CSV created successfully!
ramayana_anvaya_nllb_bleu_results.csv


In [ ]:
import os

print("File exists:", os.path.exists(OUTPUT_FILE))
print("File size:", os.path.getsize(OUTPUT_FILE), "bytes")

File exists: True
File size: 1563258 bytes


In [ ]:
check_df = pd.read_csv(OUTPUT_FILE)
print("Rows in CSV:", len(check_df))
print("Columns:", check_df.columns.tolist())

Rows in CSV: 1829
Columns: ['sample_id', 'sanskrit_sloka', 'sanskrit_prose', 'english_sloka', 'english_prose', 'bleu_score']


In [ ]:
from google.colab import files

files.download(OUTPUT_FILE)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

TASK 2 COMPLETED TILL HERE FROM HERE TASK 3 STARTED

In [ ]:
SLOKA_COLUMN = "sloka"
PROSE_COLUMN = "prose"

In [ ]:
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("Model device:", next(model.parameters()).device)
print("Tokenizer:", type(tokenizer).__name__)

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
Model device: cuda:0
Tokenizer: NllbTokenizer


In [ ]:
train_dataset = dataset["train"]

print(train_dataset)
print("Number of training samples:", len(train_dataset))
print("Columns:", train_dataset.column_names)


Dataset({
    features: ['sloka', 'prose'],
    num_rows: 16447
})
Number of training samples: 16447
Columns: ['sloka', 'prose']


In [ ]:
print(train_dataset[0])

{'sloka': 'कृत्वा निश्शब्दमेकाग्रा श्श्रुण्वन्तु हरयो ममतत्वं सङ्कीर्तयिष्यामि यथा जानामि मैथिलीम्', 'prose': 'हरयः मैथिलीम् यथा जानामि तत्वम् सङ्कीर्तयिष्यामि निश्शब्दम् कृत्वा एकाग्राः मम श्रुण्वन्तु'}


In [ ]:
SLOKA_COLUMN = "sloka"
PROSE_COLUMN = "prose"

print("Sloka column:", SLOKA_COLUMN)
print("Prose column:", PROSE_COLUMN)

Sloka column: sloka
Prose column: prose


In [ ]:
print("First Sloka:")
print(train_dataset[0][SLOKA_COLUMN])

print("\nFirst Prose:")
print(train_dataset[0][PROSE_COLUMN])

First Sloka:
कृत्वा निश्शब्दमेकाग्रा श्श्रुण्वन्तु हरयो ममतत्वं सङ्कीर्तयिष्यामि यथा जानामि मैथिलीम्

First Prose:
हरयः मैथिलीम् यथा जानामि तत्वम् सङ्कीर्तयिष्यामि निश्शब्दम् कृत्वा एकाग्राः मम श्रुण्वन्तु


In [ ]:
import os

TASK3_DIR = "/content/task3_ramayana"

os.makedirs(TASK3_DIR, exist_ok=True)

TRANSLATION_FILE = os.path.join(
    TASK3_DIR,
    "task3_train_translations.csv"
)

METRIC_FILE = os.path.join(
    TASK3_DIR,
    "task3_train_metrics.csv"
)

FINAL_FILE = os.path.join(
    TASK3_DIR,
    "ramayana_anvaya_task3_train_results.csv"
)

print("Task-3 directory:", TASK3_DIR)

Task-3 directory: /content/task3_ramayana


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
TASK3_DIR = "/content/drive/MyDrive/Ramayana_Task3"

os.makedirs(TASK3_DIR, exist_ok=True)

TRANSLATION_FILE = os.path.join(
    TASK3_DIR,
    "task3_train_translations.csv"
)

METRIC_FILE = os.path.join(
    TASK3_DIR,
    "task3_train_metrics.csv"
)

FINAL_FILE = os.path.join(
    TASK3_DIR,
    "ramayana_anvaya_task3_train_results.csv"
)

print(TASK3_DIR)

/content/drive/MyDrive/Ramayana_Task3


In [ ]:
train_df = pd.DataFrame({
    "sample_id": range(1, len(train_dataset) + 1),

    "sanskrit_sloka":
        train_dataset[SLOKA_COLUMN],

    "sanskrit_prose":
        train_dataset[PROSE_COLUMN]
})

train_df["english_sloka"] = ""
train_df["english_prose"] = ""

print("Training samples:", len(train_df))
print(train_df.head())

Training samples: 16447
   sample_id                                     sanskrit_sloka  \
0          1  कृत्वा निश्शब्दमेकाग्रा श्श्रुण्वन्तु हरयो ममत...   
1          2  कामं वा स्वयमेवाद्य तत्र मां नेतुमर्हसियत्रासौ...   
2          3  अथतान्सचिवांस्तत्रसर्वानाभाष्यरावणः सभांसन्नाद...   
3          4  तं मत्तमातङ्गविलासगामी गच्छन्तमव्यग्रमना महात्...   
4          5  इतीव देवी बहुधा विलप्य सर्वात्मना राममनुस्मरन्...   

                                      sanskrit_prose english_sloka  \
0  हरयः मैथिलीम् यथा जानामि तत्वम् सङ्कीर्तयिष्या...                 
1  वा पुरुषव्याघ्रः मे पुत्रो असौ यत्र तपः तप्यते...                 
2  अथ महाबलः जगत्सन्तापनः क्रूरः राक्षसेश्वरः राव...                 
3  मत्तमातङ्गविलासगामी महात्मा सः लक्ष्मणः गच्छन्...                 
4  देवी इतीव बहुधा विलप्य सर्वात्मना रामम् अनुस्म...                 

  english_prose  
0                
1                
2                
3                
4                


In [ ]:
def translate_batch_fast(
    texts,
    batch_size=16,
    max_length=256
):

    translations = []

    tokenizer.src_lang = SOURCE_LANG

    for start in tqdm(
        range(0, len(texts), batch_size),
        desc="Translating"
    ):

        batch = texts[start:start + batch_size]

        batch = [
            "" if x is None else str(x).strip()
            for x in batch
        ]

        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_length
        )

        inputs = {
            k: v.to(device)
            for k, v in inputs.items()
        }

        with torch.inference_mode():

            output = model.generate(
                **inputs,
                forced_bos_token_id=target_token_id,
                max_length=max_length,
                num_beams=4,
                early_stopping=True
            )

        decoded = tokenizer.batch_decode(
            output,
            skip_special_tokens=True
        )

        translations.extend(
            [x.strip() for x in decoded]
        )

        del inputs
        del output

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return translations

In [ ]:
test_sloka = train_dataset[0][SLOKA_COLUMN]
test_prose = train_dataset[0][PROSE_COLUMN]

test_E1 = translate_batch_fast(
    [test_sloka],
    batch_size=1
)[0]

test_E2 = translate_batch_fast(
    [test_prose],
    batch_size=1
)[0]

print("=" * 80)

print("SLOKA:")
print(test_sloka)

print("\nE1 - English Sloka:")
print(test_E1)

print("\nPROSE:")
print(test_prose)

print("\nE2 - English Prose:")
print(test_E2)

Translating:   0%|          | 0/1 [00:00<?, ?it/s]

Translating:   0%|          | 0/1 [00:00<?, ?it/s]

SLOKA:
कृत्वा निश्शब्दमेकाग्रा श्श्रुण्वन्तु हरयो ममतत्वं सङ्कीर्तयिष्यामि यथा जानामि मैथिलीम्

E1 - English Sloka:
I will speak to you in silence, and you will listen to me, and I will give you my opinion.

PROSE:
हरयः मैथिलीम् यथा जानामि तत्वम् सङ्कीर्तयिष्यामि निश्शब्दम् कृत्वा एकाग्राः मम श्रुण्वन्तु

E2 - English Prose:
Let me make a note of it, as I know it. Let me be silent, and let them hear me.


In [ ]:
if os.path.exists(TRANSLATION_FILE):

    train_df = pd.read_csv(
        TRANSLATION_FILE
    )

    print("Existing translation checkpoint found.")
    print("Rows:", len(train_df))

else:

    train_df.to_csv(
        TRANSLATION_FILE,
        index=False,
        encoding="utf-8-sig"
    )

    print("New translation checkpoint created.")

New translation checkpoint created.


In [ ]:
print(train_df.columns.tolist())
print(train_df.shape)

['sample_id', 'sanskrit_sloka', 'sanskrit_prose', 'english_sloka', 'english_prose']
(16447, 5)


In [ ]:
train_df["english_sloka"] = (
    train_df["english_sloka"]
    .fillna("")
    .astype(str)
)

train_df["english_prose"] = (
    train_df["english_prose"]
    .fillna("")
    .astype(str)
)

completed = (
    (train_df["english_sloka"].str.strip() != "") &
    (train_df["english_prose"].str.strip() != "")
)

print("Total samples:", len(train_df))
print("Completed:", completed.sum())
print("Remaining:", (~completed).sum())

Total samples: 16447
Completed: 0
Remaining: 16447


In [ ]:
CHECKPOINT_SIZE = 250
BATCH_SIZE = 16

total_samples = len(train_df)

for start in range(
    0,
    total_samples,
    CHECKPOINT_SIZE
):

    end = min(
        start + CHECKPOINT_SIZE,
        total_samples
    )

    block_completed = (
        (train_df.loc[
            start:end-1,
            "english_sloka"
        ].str.strip() != "") &
        (train_df.loc[
            start:end-1,
            "english_prose"
        ].str.strip() != "")
    )

    if block_completed.all():

        print(
            f"Skipping completed block "
            f"{start} - {end}"
        )

        continue

    print("\n" + "=" * 70)
    print(f"Processing samples {start} - {end}")
    print("=" * 70)

    # -------------------------
    # SLOKA → E1
    # -------------------------

    sloka_texts = train_df.loc[
        start:end-1,
        "sanskrit_sloka"
    ].tolist()

    print("Translating Sloka → E1...")

    E1 = translate_batch_fast(
        sloka_texts,
        batch_size=BATCH_SIZE,
        max_length=256
    )

    # -------------------------
    # PROSE → E2
    # -------------------------

    prose_texts = train_df.loc[
        start:end-1,
        "sanskrit_prose"
    ].tolist()

    print("Translating Prose → E2...")

    E2 = translate_batch_fast(
        prose_texts,
        batch_size=BATCH_SIZE,
        max_length=256
    )

    # Save translations
    train_df.loc[
        start:end-1,
        "english_sloka"
    ] = E1

    train_df.loc[
        start:end-1,
        "english_prose"
    ] = E2

    # CHECKPOINT
    train_df.to_csv(
        TRANSLATION_FILE,
        index=False,
        encoding="utf-8-sig"
    )

    print(
        f"Checkpoint saved: {start} - {end}"
    )

print("\nALL TRAINING TRANSLATIONS COMPLETED.")

NameError: name 'train_df' is not defined